In [35]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokenizer

BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [36]:
# !wget https://archive.ics.uci.edu/static/public/19/car+evaluation.zip
# !unzip car+evaluation.zip

In [37]:
# !unzip census+income.zip

In [38]:
import pandas as pd

df = pd.read_csv('car.data', sep=',', names=['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class'])
print(len(df))
df

1728


,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc
...,...,...,...,...,...,...,...
1723,low,low,5more,more,med,med,good
1724,low,low,5more,more,med,high,vgood
1725,low,low,5more,more,big,low,unacc
1726,low,low,5more,more,big,med,good


In [39]:
df.columns

Index(['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class'], dtype='object')

In [40]:
import numpy as np
import torch

DROP_P = 0.2

def concatenate_text(x_full):
    x = {}
    for i, j in x_full.items():
        x[i] = j if np.random.random() > DROP_P else '[UNK]'
    
    if x['buying'] == 'vhigh':
        b = 'very high'
    elif x['buying'] == 'high':
        b = 'high'
    elif x['buying'] == 'med':
        b = 'medium'
    elif x['buying'] == 'low':
        b = 'low'
    else:
        b = '[UNK]'

    if x['maint'] == 'vhigh':
        m = 'very high'
    elif x['maint'] == 'high':
        m = 'high'
    elif x['maint'] == 'med':
        m = 'medium'
    elif x['maint'] == 'low':
        m = 'low'
    else:
        m = '[UNK]'

    if x['doors'] == '5more':
        d = '5 or more'
    elif x['doors'] == '[UNK]':
        d = '[UNK]'
    else:
        d = x['doors']

    if x['persons'] == 'more':
        p = 'more than 4'
    elif x['persons'] == '[UNK]':
        p = '[UNK]'
    else:
        p = x['persons']

    if x['lug_boot'] == 'med':
        l = 'medium'
    elif x['lug_boot'] == '[UNK]':
        l = '[UNK]'
    else:
        l = x['lug_boot']

    if x['safety'] == 'med':
        s = 'medium'
    elif x['safety'] == '[UNK]':
        s = '[UNK]'
    else:
        s = x['safety']

    
    text = "".join([f"I have information about a car. ",
            f"The buying price of it is {b}. ",
            f"The maintenance price of it is {m}. ",
            f"It has {d} doors. ",
            f"It has places for {p} people. ",
            f"The size of luggage boot is {l}. ",
            f"The safety of the car is estimated as {s}."])

    return text

concatenate_text(df.iloc[0])

'I have information about a car. The buying price of it is very high. The maintenance price of it is very high. It has 2 doors. It has places for 2 people. The size of luggage boot is [UNK]. The safety of the car is estimated as [UNK].'

In [41]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop('class', axis =1),
                                                    df['class'],
                                                    test_size=.2,
                                                    random_state = 42)
y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})

X_train['text'] = X_train.apply(lambda x: concatenate_text(x), axis=1)
X_test['text'] = X_test.apply(lambda x: concatenate_text(x), axis=1)

X_train['label'] = y_train
X_test['label'] = y_test

/var/folders/p4/w2f9dq2d091g_4svlqrt38h80000gn/T/ipykernel_81083/2252343573.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
/var/folders/p4/w2f9dq2d091g_4svlqrt38h80000gn/T/ipykernel_81083/2252343573.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})


In [42]:
X_train['text'].iloc[0]

'I have information about a car. The buying price of it is very high. The maintenance price of it is very high. It has 5 or more doors. It has places for [UNK] people. The size of luggage boot is big. The safety of the car is estimated as high.'

In [43]:
len(X_train)

1382

In [44]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
import evaluate

# Define label mappings
# id2label = {0: "NOT-DONATE", 1: "DONATE"}
# label2id = {"NOT-DONATE": 0, "DONATE": 1}

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(X_train)
test_dataset = Dataset.from_pandas(X_test)

def tokenize_function(examples):
    # Adjust based on the structure of your dataset
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Format the datasets correctly with labels
tokenized_train_dataset = tokenized_train_dataset.map(lambda x: {'labels': x['label']})
tokenized_test_dataset = tokenized_test_dataset.map(lambda x: {'labels': x['label']})

Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

In [45]:
tokenized_train_dataset[0].keys()

dict_keys(['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'])

In [46]:
tokenized_train_dataset[0]['text']

'I have information about a car. The buying price of it is very high. The maintenance price of it is very high. It has 5 or more doors. It has places for [UNK] people. The size of luggage boot is big. The safety of the car is estimated as high.'

In [47]:
tokenized_train_dataset[0]['label']

0

In [48]:
import torch
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")
    

tensor([1.], device='mps:0')


In [49]:
from transformers import BertForSequenceClassification
from transformers import DataCollatorWithPadding
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from sklearn.metrics import roc_curve, auc
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
import sklearn.metrics as metrics

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4).to('mps')

model.dropout.p = 0

for param in model.bert.parameters():
    param.requires_grad = False

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=0,
    learning_rate = 0.01,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)

# Evaluation metric
auc = evaluate.load("roc_auc", 'multiclass')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # print(logits)
    predictions = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return auc.compute(prediction_scores=predictions, references=labels, multi_class='ovr')

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))
# fpr, tpr, threshold = roc_curve(y_test, pred[0][:, 1])
# roc_auc = metrics.auc(fpr, tpr)
# print("test roc_auc", roc_auc)
# print("")

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

# fpr_train, tpr_train, threshold_train = roc_curve(y_train, pred_train[0][:, 1])
# roc_auc_train = metrics.auc(fpr_train, tpr_train)
# print("train roc_auc", roc_auc_train)


# plt.title('Receiver Operating Characteristic')
# plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
# plt.legend(loc = 'lower right')
# plt.plot([0, 1], [0, 1],'r--')
# plt.xlim([0, 1])
# plt.ylim([0, 1])
# plt.ylabel('True Positive Rate')
# plt.xlabel('False Positive Rate')
# plt.show()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/220 [00:00<?, ?it/s]

/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 2.4672, 'grad_norm': 6.088619232177734, 'learning_rate': 0.009545454545454546, 'epoch': 0.45}
{'loss': 1.3275, 'grad_norm': 9.0850191116333, 'learning_rate': 0.00909090909090909, 'epoch': 0.91}


  0%|          | 0/6 [00:00<?, ?it/s]

...

test f1 0.5494314168316536
test precision 0.4613000768485415
test recall 0.6791907514450867
test accuracy 0.6791907514450867


/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/22 [00:00<?, ?it/s]

train f1 0.5836756847693878
train precision 0.49772922901644256
train recall 0.7054992764109985
train accuracy 0.7054992764109985


/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [50]:
for param in model.bert.parameters():
    param.requires_grad = True

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=20,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    learning_rate = 0.00001,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)

# # Evaluation metric
# auc = evaluate.load("roc_auc")

# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
#     return auc.compute(prediction_scores=predictions, references=labels)

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))
# fpr, tpr, threshold = roc_curve(y_test, pred[0][:, 1])
# roc_auc = metrics.auc(fpr, tpr)
# print("test roc_auc", roc_auc)
# print("")

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

# fpr_train, tpr_train, threshold_train = roc_curve(y_train, pred_train[0][:, 1])
# roc_auc_train = metrics.auc(fpr_train, tpr_train)
# print("train roc_auc", roc_auc_train)


# plt.title('Receiver Operating Characteristic')
# plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
# plt.legend(loc = 'lower right')
# plt.plot([0, 1], [0, 1],'r--')
# plt.xlim([0, 1])
# plt.ylim([0, 1])
# plt.ylabel('True Positive Rate')
# plt.xlabel('False Positive Rate')
# plt.show()

  0%|          | 0/440 [00:00<?, ?it/s]

/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.8288, 'grad_norm': 2.9672272205352783, 'learning_rate': 2.0000000000000002e-07, 'epoch': 0.45}
{'loss': 0.8231, 'grad_norm': 3.351217031478882, 'learning_rate': 4.0000000000000003e-07, 'epoch': 0.91}


  0%|          | 0/6 [00:00<?, ?it/s]

{'eval_loss': 0.8465116024017334, 'eval_roc_auc': 0.7286257724001478, 'eval_runtime': 2.7592, 'eval_samples_per_second': 125.401, 'eval_steps_per_second': 2.175, 'epoch': 1.0}


...

  0%|          | 0/6 [00:00<?, ?it/s]

test f1 0.7910597956663546
test precision 0.8175395413834721
test recall 0.8034682080924855
test accuracy 0.8034682080924855


/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/22 [00:00<?, ?it/s]

train f1 0.8388106278416705
train precision 0.8472156118395439
train recall 0.8473227206946454
train accuracy 0.8473227206946454


In [51]:
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score
from scipy.special import softmax

scores = []
n_iter = 1000
multi = True
y_pred = pred[0].argmax(axis=1)
for i in range(n_iter):
    y_true_boot, y_pred_boot, y_prob_boot = resample(y_test, y_pred, pred[0], random_state=i+1)
    # try:
    y_prob_boot = softmax(y_prob_boot, axis=1)
    if multi:
        auc = roc_auc_score(y_true_boot, y_prob_boot, multi_class="ovr", average="macro")
        f1 = f1_score(y_true_boot, y_pred_boot, average="macro")
        pr = precision_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
        rc = recall_score (y_true_boot, y_pred_boot, average="macro", zero_division=0)
    else:
        auc = roc_auc_score(y_true_boot, y_prob_boot)
        f1  = f1_score(y_true_boot, y_pred_boot)
        pr  = precision_score(y_true_boot, y_pred_boot, zero_division=0)
        rc  = recall_score (y_true_boot, y_pred_boot, zero_division=0)
    scores.append((auc, f1, accuracy_score(y_true_boot, y_pred_boot), pr, rc))
    # except ValueError:
    #     print(i)
    #     continue
scores = np.asarray(scores)
means, stds = scores.mean(0), scores.std(0, ddof=1)
names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]
{n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

{'ROC-AUC': '0.9466±0.0080',
 'F1': '0.6272±0.0412',
 'Accuracy': '0.8027±0.0207',
 'Precision': '0.6393±0.0399',
 'Recall': '0.6846±0.0476'}

In [53]:
DROP_P = 0.5

X_train, X_test, y_train, y_test = train_test_split(df.drop('class', axis =1),
                                                    df['class'],
                                                    test_size=.2,
                                                    random_state = 42)
y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})

X_train['text'] = X_train.apply(lambda x: concatenate_text(x), axis=1)
X_test['text'] = X_test.apply(lambda x: concatenate_text(x), axis=1)

X_train['label'] = y_train
X_test['label'] = y_test

train_dataset = Dataset.from_pandas(X_train)
test_dataset = Dataset.from_pandas(X_test)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Format the datasets correctly with labels
tokenized_train_dataset = tokenized_train_dataset.map(lambda x: {'labels': x['label']})
tokenized_test_dataset = tokenized_test_dataset.map(lambda x: {'labels': x['label']})

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4).to('mps')

model.dropout.p = 0

for param in model.bert.parameters():
    param.requires_grad = False

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=0,
    learning_rate = 0.01,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)

# Evaluation metric
auc = evaluate.load("roc_auc", 'multiclass')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # print(logits)
    predictions = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return auc.compute(prediction_scores=predictions, references=labels, multi_class='ovr')

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

/var/folders/p4/w2f9dq2d091g_4svlqrt38h80000gn/T/ipykernel_81083/21275793.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
/var/folders/p4/w2f9dq2d091g_4svlqrt38h80000gn/T/ipykernel_81083/21275793.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})


Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

...

test f1 0.5494314168316536
test precision 0.4613000768485415
test recall 0.6791907514450867
test accuracy 0.6791907514450867


/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/22 [00:00<?, ?it/s]

train f1 0.5836756847693878
train precision 0.49772922901644256
train recall 0.7054992764109985
train accuracy 0.7054992764109985


/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [54]:
for param in model.bert.parameters():
    param.requires_grad = True

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=20,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    learning_rate = 0.00001,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)


# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

  0%|          | 0/440 [00:00<?, ?it/s]

/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.8325, 'grad_norm': 2.465945243835449, 'learning_rate': 2.0000000000000002e-07, 'epoch': 0.45}
{'loss': 0.8354, 'grad_norm': 2.693402051925659, 'learning_rate': 4.0000000000000003e-07, 'epoch': 0.91}


  0%|          | 0/6 [00:00<?, ?it/s]

{'eval_loss': 0.8552194833755493, 'eval_roc_auc': 0.6542836312098073, 'eval_runtime': 2.6717, 'eval_samples_per_second': 129.503, 'eval_steps_per_second': 2.246, 'epoch': 1.0}


...

  0%|          | 0/6 [00:00<?, ?it/s]

test f1 0.6638885089752141
test precision 0.6469570663338364
test recall 0.7052023121387283
test accuracy 0.7052023121387283


/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/22 [00:00<?, ?it/s]

train f1 0.7322041464586719
train precision 0.7441336742958493
train recall 0.7568740955137482
train accuracy 0.7568740955137482


In [55]:
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score
from scipy.special import softmax

scores = []
n_iter = 1000
multi = True
y_pred = pred[0].argmax(axis=1)
for i in range(n_iter):
    y_true_boot, y_pred_boot, y_prob_boot = resample(y_test, y_pred, pred[0], random_state=i+1)
    # try:
    y_prob_boot = softmax(y_prob_boot, axis=1)
    if multi:
        auc = roc_auc_score(y_true_boot, y_prob_boot, multi_class="ovr", average="macro")
        f1 = f1_score(y_true_boot, y_pred_boot, average="macro")
        pr = precision_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
        rc = recall_score (y_true_boot, y_pred_boot, average="macro", zero_division=0)
    else:
        auc = roc_auc_score(y_true_boot, y_prob_boot)
        f1  = f1_score(y_true_boot, y_pred_boot)
        pr  = precision_score(y_true_boot, y_pred_boot, zero_division=0)
        rc  = recall_score (y_true_boot, y_pred_boot, zero_division=0)
    scores.append((auc, f1, accuracy_score(y_true_boot, y_pred_boot), pr, rc))
    # except ValueError:
    #     print(i)
    #     continue
scores = np.asarray(scores)
means, stds = scores.mean(0), scores.std(0, ddof=1)
names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]
{n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

{'ROC-AUC': '0.8362±0.0174',
 'F1': '0.3348±0.0285',
 'Accuracy': '0.7056±0.0248',
 'Precision': '0.3467±0.0292',
 'Recall': '0.3472±0.0338'}

In [56]:
DROP_P = 0.9

X_train, X_test, y_train, y_test = train_test_split(df.drop('class', axis =1),
                                                    df['class'],
                                                    test_size=.2,
                                                    random_state = 42)
y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})

X_train['text'] = X_train.apply(lambda x: concatenate_text(x), axis=1)
X_test['text'] = X_test.apply(lambda x: concatenate_text(x), axis=1)

X_train['label'] = y_train
X_test['label'] = y_test

train_dataset = Dataset.from_pandas(X_train)
test_dataset = Dataset.from_pandas(X_test)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Format the datasets correctly with labels
tokenized_train_dataset = tokenized_train_dataset.map(lambda x: {'labels': x['label']})
tokenized_test_dataset = tokenized_test_dataset.map(lambda x: {'labels': x['label']})

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4).to('mps')

model.dropout.p = 0

for param in model.bert.parameters():
    param.requires_grad = False

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=0,
    learning_rate = 0.01,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)

# Evaluation metric
auc = evaluate.load("roc_auc", 'multiclass')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # print(logits)
    predictions = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return auc.compute(prediction_scores=predictions, references=labels, multi_class='ovr')

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

/var/folders/p4/w2f9dq2d091g_4svlqrt38h80000gn/T/ipykernel_81083/1030730696.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
/var/folders/p4/w2f9dq2d091g_4svlqrt38h80000gn/T/ipykernel_81083/1030730696.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})


Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

...

test f1 0.5494314168316536
test precision 0.4613000768485415
test recall 0.6791907514450867
test accuracy 0.6791907514450867


/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/22 [00:00<?, ?it/s]

train f1 0.5836756847693878
train precision 0.49772922901644256
train recall 0.7054992764109985
train accuracy 0.7054992764109985


/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [57]:
for param in model.bert.parameters():
    param.requires_grad = True

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=20,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    learning_rate = 0.00001,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)


# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

  0%|          | 0/440 [00:00<?, ?it/s]

/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.8459, 'grad_norm': 1.946014404296875, 'learning_rate': 2.0000000000000002e-07, 'epoch': 0.45}
{'loss': 0.8487, 'grad_norm': 2.1331429481506348, 'learning_rate': 4.0000000000000003e-07, 'epoch': 0.91}


  0%|          | 0/6 [00:00<?, ?it/s]

{'eval_loss': 0.864189624786377, 'eval_roc_auc': 0.5112868241671767, 'eval_runtime': 2.7483, 'eval_samples_per_second': 125.897, 'eval_steps_per_second': 2.183, 'epoch': 1.0}


...

test f1 0.5494314168316536
test precision 0.4613000768485415
test recall 0.6791907514450867
test accuracy 0.6791907514450867


/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/22 [00:00<?, ?it/s]

train f1 0.5836756847693878
train precision 0.49772922901644256
train recall 0.7054992764109985
train accuracy 0.7054992764109985


/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [58]:
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score
from scipy.special import softmax

scores = []
n_iter = 1000
multi = True
y_pred = pred[0].argmax(axis=1)
for i in range(n_iter):
    y_true_boot, y_pred_boot, y_prob_boot = resample(y_test, y_pred, pred[0], random_state=i+1)
    # try:
    y_prob_boot = softmax(y_prob_boot, axis=1)
    if multi:
        auc = roc_auc_score(y_true_boot, y_prob_boot, multi_class="ovr", average="macro")
        f1 = f1_score(y_true_boot, y_pred_boot, average="macro")
        pr = precision_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
        rc = recall_score (y_true_boot, y_pred_boot, average="macro", zero_division=0)
    else:
        auc = roc_auc_score(y_true_boot, y_prob_boot)
        f1  = f1_score(y_true_boot, y_pred_boot)
        pr  = precision_score(y_true_boot, y_pred_boot, zero_division=0)
        rc  = recall_score (y_true_boot, y_pred_boot, zero_division=0)
    scores.append((auc, f1, accuracy_score(y_true_boot, y_pred_boot), pr, rc))
    # except ValueError:
    #     print(i)
    #     continue
scores = np.asarray(scores)
means, stds = scores.mean(0), scores.std(0, ddof=1)
names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]
{n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

{'ROC-AUC': '0.5561±0.0250',
 'F1': '0.2022±0.0044',
 'Accuracy': '0.6795±0.0246',
 'Precision': '0.1699±0.0061',
 'Recall': '0.2500±0.0000'}

In [59]:
DROP_P = 0

X_train, X_test, y_train, y_test = train_test_split(df.drop('class', axis =1),
                                                    df['class'],
                                                    test_size=.2,
                                                    random_state = 42)
y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})

X_train['text'] = X_train.apply(lambda x: concatenate_text(x), axis=1)
X_test['text'] = X_test.apply(lambda x: concatenate_text(x), axis=1)

X_train['label'] = y_train
X_test['label'] = y_test

train_dataset = Dataset.from_pandas(X_train)
test_dataset = Dataset.from_pandas(X_test)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Format the datasets correctly with labels
tokenized_train_dataset = tokenized_train_dataset.map(lambda x: {'labels': x['label']})
tokenized_test_dataset = tokenized_test_dataset.map(lambda x: {'labels': x['label']})

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4).to('mps')

model.dropout.p = 0

for param in model.bert.parameters():
    param.requires_grad = False

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=0,
    learning_rate = 0.01,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)

# Evaluation metric
auc = evaluate.load("roc_auc", 'multiclass')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # print(logits)
    predictions = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return auc.compute(prediction_scores=predictions, references=labels, multi_class='ovr')

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

/var/folders/p4/w2f9dq2d091g_4svlqrt38h80000gn/T/ipykernel_81083/3202789476.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
/var/folders/p4/w2f9dq2d091g_4svlqrt38h80000gn/T/ipykernel_81083/3202789476.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})


Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

...

test f1 0.5494314168316536
test precision 0.4613000768485415
test recall 0.6791907514450867
test accuracy 0.6791907514450867


/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/22 [00:00<?, ?it/s]

train f1 0.5836756847693878
train precision 0.49772922901644256
train recall 0.7054992764109985
train accuracy 0.7054992764109985


/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [60]:
for param in model.bert.parameters():
    param.requires_grad = True

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=20,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    learning_rate = 0.00001,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)


# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

  0%|          | 0/440 [00:00<?, ?it/s]

/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.8178, 'grad_norm': 2.600766897201538, 'learning_rate': 2.0000000000000002e-07, 'epoch': 0.45}
{'loss': 0.8163, 'grad_norm': 4.129016399383545, 'learning_rate': 4.0000000000000003e-07, 'epoch': 0.91}


  0%|          | 0/6 [00:00<?, ?it/s]

{'eval_loss': 0.8346763849258423, 'eval_roc_auc': 0.7813176064286353, 'eval_runtime': 2.6755, 'eval_samples_per_second': 129.323, 'eval_steps_per_second': 2.243, 'epoch': 1.0}


...

  0%|          | 0/6 [00:00<?, ?it/s]

test f1 0.8922453915445729
test precision 0.9148805703189852
test recall 0.8959537572254336
test accuracy 0.8959537572254336


/Users/barebyxlol/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/22 [00:00<?, ?it/s]

train f1 0.9450113800779338
train precision 0.9501315670142522
train recall 0.947178002894356
train accuracy 0.947178002894356


In [61]:
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score
from scipy.special import softmax

scores = []
n_iter = 1000
multi = True
y_pred = pred[0].argmax(axis=1)
for i in range(n_iter):
    y_true_boot, y_pred_boot, y_prob_boot = resample(y_test, y_pred, pred[0], random_state=i+1)
    # try:
    y_prob_boot = softmax(y_prob_boot, axis=1)
    if multi:
        auc = roc_auc_score(y_true_boot, y_prob_boot, multi_class="ovr", average="macro")
        f1 = f1_score(y_true_boot, y_pred_boot, average="macro")
        pr = precision_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
        rc = recall_score (y_true_boot, y_pred_boot, average="macro", zero_division=0)
    else:
        auc = roc_auc_score(y_true_boot, y_prob_boot)
        f1  = f1_score(y_true_boot, y_pred_boot)
        pr  = precision_score(y_true_boot, y_pred_boot, zero_division=0)
        rc  = recall_score (y_true_boot, y_pred_boot, zero_division=0)
    scores.append((auc, f1, accuracy_score(y_true_boot, y_pred_boot), pr, rc))
    # except ValueError:
    #     print(i)
    #     continue
scores = np.asarray(scores)
means, stds = scores.mean(0), scores.std(0, ddof=1)
names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]
{n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

{'ROC-AUC': '0.9975±0.0010',
 'F1': '0.7792±0.0409',
 'Accuracy': '0.8961±0.0158',
 'Precision': '0.8201±0.0337',
 'Recall': '0.8210±0.0328'}